## Downloading flood conditioning variables

## Elevation, slope

In [66]:
import ee
import numpy as np
import time

# Initialize Earth Engine
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

def tile_name_from_latlon(lat, lon):

    """
    Example usage:
    Nairobi coordinates: 0.0236° S, 37.9062° E
    print(f"Nairobi tile: {tile_name_from_latlon(lat = 1.29, lon = 36.82)}")
    """

    # Data is in 3 x 3 degree tiles so
     
    lat_tile = (lat // 3) * 3
    lon_tile = (lon // 3) * 3
    lat_tile, lon_tile = int(lat_tile), int(lon_tile)

    lat_prefix = 'N' if lat_tile >= 0 else 'S'
    lon_prefix = 'E' if lon_tile >= 0 else 'W' 

    return f"{lat_prefix}{abs(lat_tile):02d}{lon_prefix}{abs(lon_tile):03d}"

def get_country_bonds(country_name):
    """
    Returns the bounding box of a country as [min_lon, min_lat, max_lon, max_lat]
    """
    countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    country = countries.filter(ee.Filter.eq('country_na', country_name))
    bounds = np.array(country.geometry().bounds().getInfo()['coordinates'][0])

    # Get min and max latitudes and longitudes
    min_lon, max_lon = bounds[:, 0].min(), bounds[:, 0].max()
    min_lat, max_lat = bounds[:, 1].min(), bounds[:, 1].max()
    return [min_lon, min_lat, max_lon, max_lat]

def generate_tiles_from_bounds(bounds, tile_size=3):
    """
    Generates a list of tile names that cover the given bounding box.
    """
    min_lon, min_lat, max_lon, max_lat = bounds

    # Align the bounds to the tile grid
    start_lon = (min_lon // tile_size) * tile_size
    end_lon = (max_lon // tile_size) * tile_size
    start_lat = (min_lat // tile_size) * tile_size
    end_lat = (max_lat // tile_size) * tile_size

    tiles = []
    for lat in range(int(start_lat), int(end_lat) + tile_size, tile_size):
        for lon in range(int(start_lon), int(end_lon) + tile_size, tile_size):
            tile = (lon, lat, lon + tile_size, lat + tile_size)
            tiles.append(tile)
            
    return tiles

def download_data_for_tile(tile, image, dataset_name, export_params):
    """
    Exports data for a given tile and dataset to Google Drive.
    """
    min_lon, min_lat, max_lon, max_lat = tile
    tile_name = tile_name_from_latlon(min_lat, min_lon)
    

    tile_geom = ee.Geometry.Rectangle(tile)
    export_params['region'] = tile_geom

    # Check if a task with the same description already exists
    tasks = ee.batch.Task.list()

    task_exists = any(
    ((t.config.get('description') == f"{tile_name}_{dataset_name}") and
    (t.state in ['READY', 'RUNNING'])) or
    ((t.config.get('description') == f"{tile_name}_{dataset_name}") and
    (t.state == 'COMPLETED'))
    for t in tasks
    )

    if task_exists:
        print(f"Task for {tile_name} and dataset {dataset_name} already exists. Skipping export.")
        return True

    print(f"Exporting tile: {tile_name} for dataset: {dataset_name}")
    task = ee.batch.Export.image.toDrive(
        image = image,
        description = f"{tile_name}_{dataset_name}",
        **export_params
    )
    task.start()
    return True

def check_task_status(n = 50):
    tasks = ee.batch.Task.list()
    print(f"{'TASK DESCRIPTION':<30} | {'STATE':<10} | {'ID'}")
    print("-" * 60)
    for task in tasks[:n]:  # Check the first n tasks
        status = task.status()
        description = status.get('description', 'No Description')
        state = status.get('state', 'UNKNOWN')
        task_id = status.get('id')
        print(f"{description:<30} | {state:<10} | {task_id}")
    

In [67]:
# Get the bounding box for Kenya
kenya_bounds = get_country_bonds("Kenya")

# Generate 3x3 degree tiles that cover Kenya
kenya_tiles = generate_tiles_from_bounds(kenya_bounds, tile_size=3)

# Define global datasets
# DEM: SRTM V3 (30m)
srtm = ee.Image("USGS/SRTMGL1_003")
dem_global = srtm.select('elevation')

# Slope: Derived from SRTM
slope_global = ee.Terrain.slope(dem_global)

# Define export parameters
export_params = {
    'scale': 30, # Force 30m resolution for ALL layers
    'crs': 'EPSG:4326',
    'maxPixels': 1e13,
    'fileFormat': 'GeoTIFF',
    'folder': 'Kenya_Flood_Data_3x3' # Creates this folder in Drive
}

# Download DEM and Slope for each tile
for tile in kenya_tiles:
    download_data_for_tile(tile, dem_global, "DEM", export_params)
    download_data_for_tile(tile, slope_global, "Slope", export_params)
    time.sleep(10)

# Check task status
check_task_status()


Task for S06E033 and dataset DEM already exists. Skipping export.
Exporting tile: S06E033 for dataset: Slope
Task for S06E036 and dataset DEM already exists. Skipping export.
Exporting tile: S06E036 for dataset: Slope
Task for S06E039 and dataset DEM already exists. Skipping export.
Exporting tile: S06E039 for dataset: Slope
Task for S03E033 and dataset DEM already exists. Skipping export.
Exporting tile: S03E033 for dataset: Slope
Task for S03E036 and dataset DEM already exists. Skipping export.
Exporting tile: S03E036 for dataset: Slope
Task for S03E039 and dataset DEM already exists. Skipping export.
Exporting tile: S03E039 for dataset: Slope
Task for N00E033 and dataset DEM already exists. Skipping export.
Exporting tile: N00E033 for dataset: Slope
Task for N00E036 and dataset DEM already exists. Skipping export.
Exporting tile: N00E036 for dataset: Slope
Task for N00E039 and dataset DEM already exists. Skipping export.
Exporting tile: N00E039 for dataset: Slope
Task for N03E033 an

In [69]:
check_task_status()

TASK DESCRIPTION               | STATE      | ID
------------------------------------------------------------
N03E039_Slope                  | COMPLETED  | LVLSDOJSAWN7PP3YJEM4H25M
N03E036_Slope                  | COMPLETED  | ZX4UWLQQVHCCR72CNRVZACWZ
N03E033_Slope                  | COMPLETED  | BMBAOQEKKYVXNSRAYR4K3YR2
N00E039_Slope                  | COMPLETED  | VM6YX4MNA7CRVM7UR3NJMFWT
N00E036_Slope                  | COMPLETED  | LZWBBVEBVVYANXSUGMDUWOVG
N00E033_Slope                  | COMPLETED  | TG4CEIF37TNCIT6DJHVFF62C
S03E039_Slope                  | COMPLETED  | 7H7B5ODYGZCZ3SO2434XAG67
S03E036_Slope                  | COMPLETED  | F6MAOBIFPNY2FNWR5T76XWSJ
S03E033_Slope                  | COMPLETED  | HJ2AW5KKV76MJV5I3PW3YCLR
S06E039_Slope                  | COMPLETED  | O7TTADZU2TE7CEDHI5LRBFC5
S06E036_Slope                  | COMPLETED  | GJEUSOAFAZUVVTLOBFGDF7PU
S06E033_Slope                  | COMPLETED  | SQ2X5YDG5N4IX6WYNU6V2QAW
N03E039_DEM                    | COMPL